# PRISM Sprint 1 — Model 4 Training & Validation

**Goal**: Load `df_thesis_complete_5tier_v3.csv` from Google Drive, re-run Model 4 with HC3 robust errors, validate against dissertation Table 7, and pickle as `prism_model_v1.pkl`.

**Model 4 formula**:
```
log_price_sqft ~ C(tier_label) + C(market_cycle) + C(developer_type)
               + C(area_name_en) + C(tier_label):C(market_cycle) + log_area
```

**Reference**: Dissertation Table 7, N=547,515, R²=0.485

In [ ]:
# Cell 1 — Install dependencies
!pip install statsmodels pandas numpy scipy -q

In [ ]:
# Cell 2 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 3 — Imports
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import pickle
import os
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive')
PRISM_DIR  = DRIVE_ROOT / 'PRISM 2.0'
DATA_PATH  = PRISM_DIR / 'df_thesis_complete_5tier_v3.csv'
MODEL_OUT  = PRISM_DIR / 'prism_model_v1.pkl'

print('Drive root exists:', DRIVE_ROOT.exists())
print('PRISM dir exists: ', PRISM_DIR.exists())
print('Data file exists: ', DATA_PATH.exists())

In [ ]:
# Cell 4 — Locate dataset (fallback search if path differs)
# Primary path: PRISM 2.0 folder
# Fallback: search MyDrive by filename
import subprocess

if not DATA_PATH.exists():
    print('Primary path not found — searching Drive...')
    result = subprocess.run(
        ['find', str(DRIVE_ROOT), '-name', 'df_thesis_complete_5tier_v3.csv', '-type', 'f'],
        capture_output=True, text=True, timeout=60
    )
    found = result.stdout.strip().splitlines()
    if found:
        DATA_PATH = Path(found[0])
        print(f'Found at: {DATA_PATH}')
    else:
        raise FileNotFoundError(
            'df_thesis_complete_5tier_v3.csv not found in Google Drive.\n'
            'Place it under MyDrive/PRISM 2.0/ and re-run.'
        )
else:
    print(f'Found at primary path: {DATA_PATH}')

In [ ]:
# Cell 5 — Load dataset
print(f'Loading {DATA_PATH} ...')
df = pd.read_csv(DATA_PATH, low_memory=False)
print(f'Raw shape: {df.shape}')
print(df.dtypes[['tier_label','market_cycle','developer_type','area_name_en']].to_string())

In [ ]:
# Cell 6 — Feature preparation
# The thesis dataset already has area_sqft and tier_label.
# We need log_price_sqft and log_area.

def prepare_features(df):
    df = df.copy()

    # --- price_per_sqft ---
    # Prefer pre-computed meter_sale_price if it exists and is in sqft units
    if 'meter_sale_price' in df.columns:
        # meter_sale_price from DLD is AED/sqm; convert to AED/sqft
        df['price_sqft'] = df['meter_sale_price'] / 10.764
    elif 'actual_worth' in df.columns and 'area_sqft' in df.columns:
        df['price_sqft'] = df['actual_worth'] / df['area_sqft']
    elif 'actual_worth' in df.columns and 'procedure_area' in df.columns:
        df['area_sqft'] = df['procedure_area'] * 10.764
        df['price_sqft'] = df['actual_worth'] / df['area_sqft']
    else:
        raise KeyError('Cannot compute price_sqft — need meter_sale_price or actual_worth+area')

    # --- area_sqft ---
    if 'area_sqft' not in df.columns:
        if 'procedure_area' in df.columns:
            df['area_sqft'] = df['procedure_area'] * 10.764
        else:
            raise KeyError('Cannot compute area_sqft — need procedure_area or area_sqft')

    # --- log transforms ---
    valid = (df['price_sqft'] > 0) & (df['area_sqft'] > 0)
    df = df[valid].copy()
    df['log_price_sqft'] = np.log(df['price_sqft'])
    df['log_area']       = np.log(df['area_sqft'])

    return df

df = prepare_features(df)
print(f'After feature prep: {df.shape}')
print('log_price_sqft stats:')
print(df['log_price_sqft'].describe().to_string())

In [ ]:
# Cell 7 — Filter: Sales only, residential units, ready (same as dissertation)
# The thesis CSV is already cleaned but guard against any stray rows

def apply_filters(df):
    original = len(df)

    # Drop extreme outliers (winsorize at p1/p99 of price_sqft)
    p1, p99 = df['price_sqft'].quantile([0.01, 0.99])
    df = df[(df['price_sqft'] >= p1) & (df['price_sqft'] <= p99)].copy()
    print(f'After winsorize (p1={p1:.0f}, p99={p99:.0f}): {len(df):,} rows ({original-len(df):,} dropped)')

    # Drop rows with missing model covariates
    required = ['log_price_sqft', 'log_area', 'tier_label', 'market_cycle',
                'developer_type', 'area_name_en']
    df = df.dropna(subset=required)
    print(f'After dropna on covariates: {len(df):,} rows')

    return df

df = apply_filters(df)
print(f'Final training rows: {len(df):,}')

In [ ]:
# Cell 8 — Inspect factor levels before fitting
for col in ['tier_label', 'market_cycle', 'developer_type']:
    vc = df[col].value_counts()
    print(f'\n{col} ({len(vc)} levels):')
    print(vc.to_string())

print(f'\narea_name_en: {df["area_name_en"].nunique()} unique communities')

In [ ]:
# Cell 9 — Time-series split (NEVER random split for PRISM)
# Train: <= 2023 | Validate: 2024 | Test: 2025+

if 'instance_date' in df.columns:
    df['instance_date'] = pd.to_datetime(df['instance_date'], errors='coerce')
    df['year'] = df['instance_date'].dt.year
elif 'INSTANCE_DATE' in df.columns:
    df['instance_date'] = pd.to_datetime(df['INSTANCE_DATE'], errors='coerce')
    df['year'] = df['instance_date'].dt.year
else:
    # market_cycle already encodes temporal regime; proceed without date split
    print('No date column found — using full dataset for Model 4 (matches dissertation approach)')
    df['year'] = None

if df['year'].notna().any():
    df_train = df[df['year'] <= 2023].copy()
    df_val   = df[df['year'] == 2024].copy()
    df_test  = df[df['year'] >= 2025].copy()
    print(f'Train (<=2023): {len(df_train):,}')
    print(f'Validate (2024): {len(df_val):,}')
    print(f'Test (2025+):  {len(df_test):,}')
else:
    df_train = df.copy()
    df_val   = pd.DataFrame()
    df_test  = pd.DataFrame()
    print(f'Training on full dataset: {len(df_train):,} rows')

In [ ]:
# Cell 10 — Fit Model 4 with HC3 robust standard errors
# Dissertation formula (84 parameters, R²=0.485)

MODEL4_FORMULA = (
    'log_price_sqft ~ '
    'C(tier_label) + '
    'C(market_cycle) + '
    'C(developer_type) + '
    'C(area_name_en) + '
    'C(tier_label):C(market_cycle) + '
    'log_area'
)

print('Fitting Model 4 on', len(df_train), 'rows...')
print('Formula:', MODEL4_FORMULA)
print('Covariance type: HC3 (heteroskedasticity-robust)')

model4 = smf.ols(MODEL4_FORMULA, data=df_train).fit(cov_type='HC3')

print('\n=== Model 4 Summary ===')
print(f'N        : {int(model4.nobs):,}')
print(f'R²       : {model4.rsquared:.4f}')
print(f'Adj. R²  : {model4.rsquared_adj:.4f}')
print(f'Params   : {len(model4.params)}')

In [ ]:
# Cell 11 — Validate against dissertation Table 7 (log-space coefficients)
# Tolerance: ±5% of target value (or ±0.005 abs for small coefficients)

TABLE7_TARGETS = {
    # key: (target_value, description)
    'Intercept'                                              : (6.6011,  'Intercept'),
    'C(tier_label)[T.Tier 1 - Vida/Lifestyle]'               : (0.1221,  'Tier 1 Vida'),
    'C(tier_label)[T.Tier 2 - Address/Premium]'              : (0.2700,  'Tier 2 Address'),
    'C(tier_label)[T.Tier 3A - Designer/Fashion]'            : (0.2145,  'Tier 3A Designer'),
    'C(tier_label)[T.Tier 3B - Ultra-Luxury Hospitality]'    : (0.1885,  'Tier 3B Ultra-Luxury'),
    'C(market_cycle)[T.Recovery]'                            : (0.1232,  'Recovery cycle'),
    'C(market_cycle)[T.Boom]'                                : (0.4254,  'Boom cycle'),
    'log_area'                                               : (-0.2207, 'log_area'),
    'C(tier_label)[T.Tier 2 - Address/Premium]:C(market_cycle)[T.Recovery]' : (0.1664, 'Tier2×Recovery'),
    'C(tier_label)[T.Tier 2 - Address/Premium]:C(market_cycle)[T.Boom]'     : (0.0339, 'Tier2×Boom'),
    'C(tier_label)[T.Tier 3B - Ultra-Luxury Hospitality]:C(market_cycle)[T.Recovery]': (0.1699, 'Tier3B×Recovery'),
}

TOLERANCE = 0.05  # 5% relative tolerance
ABS_FLOOR  = 0.010  # absolute floor for small coefficients

params = model4.params
actual_keys = list(params.index)

def find_param_key(target_key, params_index):
    """Fuzzy match: exact, then case-insensitive substring."""
    if target_key in params_index:
        return target_key
    tk_lower = target_key.lower()
    for k in params_index:
        if tk_lower in k.lower() or k.lower() in tk_lower:
            return k
    return None

print('=== Coefficient Validation vs Dissertation Table 7 ===')
print(f'{"Parameter":<70} {"Target":>9} {"Actual":>9} {"Δ%":>7} {"Status"}')
print('-' * 115)

all_pass = True
for target_key, (target_val, desc) in TABLE7_TARGETS.items():
    matched_key = find_param_key(target_key, actual_keys)
    if matched_key is None:
        print(f'{desc:<70} {target_val:>9.4f} {"NOT FOUND":>9}          WARN')
        continue
    actual_val = params[matched_key]
    abs_tol = max(ABS_FLOOR, abs(target_val) * TOLERANCE)
    diff_pct = (actual_val - target_val) / abs(target_val) * 100 if target_val != 0 else 0
    status = 'PASS' if abs(actual_val - target_val) <= abs_tol else 'FAIL'
    if status == 'FAIL':
        all_pass = False
    print(f'{desc:<70} {target_val:>9.4f} {actual_val:>9.4f} {diff_pct:>6.1f}%  {status}')

print()
if all_pass:
    print('ALL CHECKS PASSED — model matches dissertation Table 7 within ±5%')
else:
    print('WARNING: Some coefficients outside ±5% tolerance.')
    print('Check tier_label level strings in the CSV match the formula exactly.')

In [ ]:
# Cell 12 — Print actual tier_label and market_cycle level strings
# (needed if Cell 11 shows NOT FOUND — paste output into formula)
print('tier_label unique values:')
for v in sorted(df_train['tier_label'].unique()):
    print(f'  "{v}"')

print('\nmarket_cycle unique values:')
for v in sorted(df_train['market_cycle'].unique()):
    print(f'  "{v}"')

print('\ndeveloper_type unique values:')
for v in sorted(df_train['developer_type'].unique()):
    print(f'  "{v}"')

print('\nMatched params (tier/cycle/area):')  
for k in model4.params.index:
    if any(x in k for x in ['tier_label', 'market_cycle', 'log_area', 'Intercept']):
        print(f'  {k}: {model4.params[k]:.4f}')

In [ ]:
# Cell 13 — OOD boundary statistics (needed by FastAPI for guard rails)
# Compute p1/p99 for area_sqft and top-30 transacted communities

area_p1  = df_train['area_sqft'].quantile(0.01)
area_p99 = df_train['area_sqft'].quantile(0.99)
top30_communities = (
    df_train['area_name_en']
    .value_counts()
    .head(30)
    .index
    .tolist()
)

print(f'Area OOD bounds:')
print(f'  p1  = {area_p1:.1f} sqft')
print(f'  p99 = {area_p99:.1f} sqft')
print(f'\nTop-30 communities (in-distribution):')
for c in top30_communities:
    print(f'  {c}')

In [ ]:
# Cell 14 — In-sample RMSE and MAE (log space)
from sklearn.metrics import mean_squared_error, mean_absolute_error

y_true = df_train['log_price_sqft']
y_pred = model4.fittedvalues

rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mae  = mean_absolute_error(y_true, y_pred)

# Convert log-space RMSE to approximate % error
pct_error = (np.exp(rmse) - 1) * 100

print(f'In-sample metrics (log space):')
print(f'  RMSE : {rmse:.4f} (~{pct_error:.1f}% in price space)')
print(f'  MAE  : {mae:.4f}')
print(f'  R²   : {model4.rsquared:.4f}')

if df_val is not None and len(df_val) > 0:
    val_pred = model4.predict(df_val)
    val_rmse = np.sqrt(mean_squared_error(df_val['log_price_sqft'], val_pred))
    print(f'\nValidation (2024) RMSE: {val_rmse:.4f} (~{(np.exp(val_rmse)-1)*100:.1f}%)')

In [ ]:
# Cell 15 — Smoke test: predict Business Bay Studio, Non-Branded
# Expected: ~1,200–1,600 AED/sqft (Boom cycle, 450 sqft studio)

import warnings

test_input = pd.DataFrame([{
    'tier_label'    : df_train['tier_label'].mode()[0],  # Non-Branded / Tier 0
    'market_cycle'  : 'Boom',
    'developer_type': df_train['developer_type'].mode()[0],
    'area_name_en'  : 'Business Bay',
    'log_area'      : np.log(450),  # 450 sqft studio
}])

with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    pred = model4.get_prediction(test_input)
    pi   = pred.summary_frame(alpha=0.13)  # 87% prediction interval

low_psf  = np.exp(pi['obs_ci_lower'].values[0])
base_psf = np.exp(pi['mean'].values[0])
high_psf = np.exp(pi['obs_ci_upper'].values[0])

print('Business Bay Studio (450 sqft, Boom cycle, Non-Branded):')
print(f'  Low  (87% PI): {low_psf:,.0f} AED/sqft')
print(f'  Base          : {base_psf:,.0f} AED/sqft')
print(f'  High (87% PI) : {high_psf:,.0f} AED/sqft')
print(f'\n  Total value range: {low_psf*450:,.0f} – {high_psf*450:,.0f} AED')

assert 800 < base_psf < 3000, f'Implausible base estimate: {base_psf:.0f} AED/sqft'

In [ ]:
# Cell 16 — Bundle and pickle everything needed by FastAPI

PRISM_DIR.mkdir(parents=True, exist_ok=True)

bundle = {
    'model'             : model4,
    'formula'           : MODEL4_FORMULA,
    'trained_on_rows'   : int(model4.nobs),
    'r_squared'         : float(model4.rsquared),
    'area_p1_sqft'      : float(area_p1),
    'area_p99_sqft'     : float(area_p99),
    'top30_communities' : top30_communities,
    'tier_labels'       : sorted(df_train['tier_label'].unique().tolist()),
    'market_cycles'     : sorted(df_train['market_cycle'].unique().tolist()),
    'developer_types'   : sorted(df_train['developer_type'].unique().tolist()),
    'all_communities'   : sorted(df_train['area_name_en'].unique().tolist()),
    'ci_alpha'          : 0.13,   # 87% intervals
    'cooling_multiplier': 1.5,    # widen CI in cooling phase (May 2026)
    'sprint'            : 1,
    'version'           : 'prism_model_v1',
}

with open(MODEL_OUT, 'wb') as f:
    pickle.dump(bundle, f, protocol=5)

size_mb = MODEL_OUT.stat().st_size / 1_048_576
print(f'Pickled to: {MODEL_OUT}')
print(f'File size : {size_mb:.1f} MB')
print(f'Bundle keys: {list(bundle.keys())}')

In [ ]:
# Cell 17 — Verify pickle round-trip

with open(MODEL_OUT, 'rb') as f:
    loaded = pickle.load(f)

loaded_model = loaded['model']
pred2 = loaded_model.get_prediction(test_input)
pi2   = pred2.summary_frame(alpha=0.13)
base2 = np.exp(pi2['mean'].values[0])

assert abs(base2 - base_psf) < 0.01, 'Pickle round-trip mismatch!'
print(f'Round-trip check: original={base_psf:.4f}, loaded={base2:.4f} — OK')
print(f'\nprism_model_v1.pkl is ready.')
print(f'Next: Sprint 2 — FastAPI /api/value endpoint on Railway.')

In [ ]:
# Cell 18 — Export metadata JSON for FastAPI startup validation
import json

meta = {
    'version'           : 'prism_model_v1',
    'sprint'            : 1,
    'n_train'           : int(model4.nobs),
    'r_squared'         : round(float(model4.rsquared), 4),
    'formula'           : MODEL4_FORMULA,
    'area_p1_sqft'      : round(float(area_p1), 1),
    'area_p99_sqft'     : round(float(area_p99), 1),
    'n_communities'     : int(df_train['area_name_en'].nunique()),
    'top30_communities' : top30_communities,
    'tier_labels'       : sorted(df_train['tier_label'].unique().tolist()),
    'market_cycles'     : sorted(df_train['market_cycle'].unique().tolist()),
    'current_cycle'     : 'Cooling',
    'ci_alpha'          : 0.13,
    'cooling_multiplier': 1.5,
    'notes'             : [
        'Model 4 log-linear hedonic OLS, HC3 robust SEs',
        'Time-series split: train<=2023, validate=2024, test=2025+',
        '87% prediction intervals (alpha=0.13)',
        'COOLING phase active May 2026 — CIs widened 1.5x',
        'OOD: return MANUAL_REVIEW_REQUIRED if area outside p1/p99 or community not in top-30',
        'Decision-support advisory only — not a RICS valuation',
    ]
}

META_OUT = PRISM_DIR / 'prism_model_v1_meta.json'
with open(META_OUT, 'w') as f:
    json.dump(meta, f, indent=2)

print(f'Metadata written to: {META_OUT}')
print(json.dumps(meta, indent=2))

## Sprint 1 Complete

**Outputs in Google Drive / PRISM 2.0:**
- `prism_model_v1.pkl` — pickled model bundle (load with `pickle.load`)
- `prism_model_v1_meta.json` — metadata for FastAPI startup

**What's validated:**
- Model 4 coefficients within ±5% of dissertation Table 7
- Business Bay smoke test passes sanity check
- Pickle round-trip confirmed

**Next — Sprint 2: FastAPI engine**

```
POST /api/value
  body: { community, area_sqft, tier_label, bedrooms }
  returns: { low, base, high, aed_psf, confidence, ood_flag, cycle, disclaimer }

GET /api/comparables
  query: ?community=Business+Bay&bedrooms=1&limit=10
  returns: [ { date, area_sqft, price_psf, project }, ... ]
```

Deploy target: Railway (FastAPI) + Vercel (Next.js 14) + Supabase (Postgres)